# Component-wise Energy Experiment (B1): single-shot / judge / retry decomposition
Same models, prompts, parser, scoring and quantization as v4. Differences:
- Only tau = 7; first **50** samples per task (same seed and order as v4, i.e. sample_idx 0-49).
- Every model call is measured in its **own** CodeCarbon interval: `gen1`, `judge1`, `gen2`, `judge2`, `gen3`, `judge3`.
- Input and output token counts are stored for every call.
- Each sample is repeated **REPEATS = 3** times to estimate measurement variability.
- A warm-up call precedes measurement for every model.
- Results: `agentic_energy_results_v4/component_run/components.csv` (one row per call), resume-safe.


## 1. GPU check

In [ ]:
import torch
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(f"Total memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 2. Install

In [ ]:
!pip install -q transformers accelerate bitsandbytes codecarbon datasets evaluate rouge_score sentencepiece
!pip uninstall -y -q hf_xet 2>/dev/null || true

## 3. Drive + cache setup (run BEFORE importing transformers)

In [ ]:
import sys, os, tempfile
sys.modules['hf_xet'] = None
from google.colab import drive
drive.mount('/content/drive')

DRIVE_CACHE_DIR = "/content/drive/MyDrive/hf_cache"
os.makedirs(DRIVE_CACHE_DIR, exist_ok=True)
os.environ["HF_HOME"] = DRIVE_CACHE_DIR
os.environ["HF_HUB_DISABLE_XET"] = "1"
DRIVE_TMPDIR = os.path.join(DRIVE_CACHE_DIR, "tmp"); os.makedirs(DRIVE_TMPDIR, exist_ok=True)
for k in ("TMPDIR", "TEMP", "TMP"): os.environ[k] = DRIVE_TMPDIR
tempfile.tempdir = DRIVE_TMPDIR

RESULTS_DIR = "/content/drive/MyDrive/agentic_energy_results_v4/component_run"
os.makedirs(RESULTS_DIR, exist_ok=True)
print("Results dir:", RESULTS_DIR)
if "huggingface_hub" in sys.modules:
    print("WARNING: huggingface_hub already imported; restart runtime and run this cell first.")

## 3b. Clean stale/partial downloads and check disk (run after every reconnect)

In [ ]:
import glob, subprocess
hub_dir = os.path.join(os.environ["HF_HOME"], "hub")
removed = 0
for f in glob.glob(os.path.join(hub_dir, "**", "*.incomplete"), recursive=True):
    print("removing partial download:", f); os.remove(f); removed += 1
for sub in ("tmp", "offload"):
    d = os.path.join(DRIVE_CACHE_DIR, sub)
    for f in glob.glob(os.path.join(d, "**", "*"), recursive=True):
        if os.path.isfile(f):
            os.remove(f); removed += 1
print(f"Removed {removed} stale files.")
print("Local disk:", subprocess.run(["df", "-h", "/"], capture_output=True, text=True).stdout.splitlines()[-1])
print("Drive hf_cache size:", subprocess.run(["du", "-sh", DRIVE_CACHE_DIR], capture_output=True, text=True).stdout.strip())
print("Drive free:", subprocess.run(["df", "-h", "/content/drive/MyDrive"], capture_output=True, text=True).stdout.splitlines()[-1])

## 4. Hugging Face login (token from Colab Secrets: add a secret named HF_TOKEN via the key icon in the left sidebar)

In [ ]:
from google.colab import userdata
from huggingface_hub import login, whoami
login(userdata.get("HF_TOKEN"))
print("Logged in as:", whoami()["name"])

## 5. Imports, environment record

In [ ]:
import re, time, json, gc, shutil, platform
import pandas as pd
from datetime import datetime, timezone
import transformers, accelerate, bitsandbytes, codecarbon, datasets, evaluate
from transformers import AutoModelForCausalLM, AutoTokenizer, AutoConfig, BitsAndBytesConfig
from codecarbon import EmissionsTracker

env_info = {
    "python": platform.python_version(), "torch": torch.__version__, "cuda": torch.version.cuda,
    "transformers": transformers.__version__, "accelerate": accelerate.__version__,
    "bitsandbytes": bitsandbytes.__version__, "codecarbon": codecarbon.__version__,
    "datasets": datasets.__version__, "evaluate": evaluate.__version__,
    "gpu": torch.cuda.get_device_name(0), "recorded_at": datetime.now(timezone.utc).isoformat(),
}
with open(os.path.join(RESULTS_DIR, "env_info.json"), "w") as f: json.dump(env_info, f, indent=2)
print(json.dumps(env_info, indent=2))

## 6. Configuration

In [ ]:
MODEL_REGISTRY = {
    "llama-3.2-3b":  {"hf_id": "meta-llama/Llama-3.2-3B-Instruct",            "load_in_4bit": True, "category": "small"},
    "qwen2.5-3b":    {"hf_id": "Qwen/Qwen2.5-3B-Instruct",                     "load_in_4bit": True, "category": "small"},
    "gemma-2-2b":    {"hf_id": "google/gemma-2-2b-it",                         "load_in_4bit": True, "category": "small"},
    "llama-3-8b":    {"hf_id": "meta-llama/Meta-Llama-3-8B-Instruct",          "load_in_4bit": True, "category": "medium"},
    "qwen2.5-7b":    {"hf_id": "Qwen/Qwen2.5-7B-Instruct",                     "load_in_4bit": True, "category": "medium"},
    "mistral-7b":    {"hf_id": "mistralai/Mistral-7B-Instruct-v0.3",           "load_in_4bit": True, "category": "medium"},
    "llama-3-70b":   {"hf_id": "unsloth/llama-3-70b-Instruct-bnb-4bit",        "prequantized": True, "category": "large"},
    "qwen2.5-72b":   {"hf_id": "unsloth/Qwen2.5-72B-Instruct-bnb-4bit",        "prequantized": True, "category": "large"},
    "mistral-large": {"hf_id": "unsloth/Mistral-Large-Instruct-2407-bnb-4bit", "prequantized": True, "category": "large"},
}

MAX_EXTRA_TURNS = 2
THRESHOLDS = [7]
JUDGE_MAX_NEW_TOKENS = 8
TASK_MAX_TOKENS = {"summarization": 128, "classification": 20, "math": 512}
ACTIVE_TASKS = ["classification", "math", "summarization"]
N_SAMPLES_PER_TASK = 50
REPEATS = 3
SEED = 42
FLUSH_EVERY = 10

COMP_CSV = os.path.join(RESULTS_DIR, "components.csv")

## 7. Model loading (records device map and quantization config)

In [ ]:
MODEL_INFO_PATH = os.path.join(RESULTS_DIR, "model_info.json")

def load_model(model_key):
    cfg = MODEL_REGISTRY[model_key]
    tokenizer = AutoTokenizer.from_pretrained(cfg["hf_id"], cache_dir=DRIVE_CACHE_DIR)
    gc.collect(); torch.cuda.empty_cache()
    free_gib = torch.cuda.mem_get_info()[0] / 1024**3
    safe_gib = max(free_gib - 4, 8)
    print(f"  [{model_key}] free GPU memory {free_gib:.1f} GiB, limit {safe_gib:.0f} GiB")
    load_kwargs = {"device_map": "auto", "max_memory": {0: f"{safe_gib:.0f}GiB"},
                   "cache_dir": DRIVE_CACHE_DIR, "low_cpu_mem_usage": True}
    if cfg.get("prequantized"):
        model_config = AutoConfig.from_pretrained(cfg["hf_id"], cache_dir=DRIVE_CACHE_DIR)
        load_kwargs["config"] = model_config
    else:
        load_kwargs["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True, bnb_4bit_quant_type="nf4")
    model = AutoModelForCausalLM.from_pretrained(cfg["hf_id"], **load_kwargs)

    # --- record placement + quantization config ---
    devmap = getattr(model, "hf_device_map", {}) or {}
    devices = sorted({str(v) for v in devmap.values()})
    qc = getattr(model.config, "quantization_config", None)
    if qc is not None and not isinstance(qc, dict):
        qc = qc.to_dict() if hasattr(qc, "to_dict") else vars(qc)
    n_params = sum(p.numel() for p in model.parameters())
    info = {"hf_id": cfg["hf_id"], "category": cfg["category"], "devices_used": devices,
            "n_layers_offloaded": sum(1 for v in devmap.values() if str(v) not in ("0", "cuda:0")),
            "quantization_config": {k: str(v) for k, v in (qc or {}).items()},
            "param_count_reported": n_params, "loaded_at": datetime.now(timezone.utc).isoformat()}
    all_info = json.load(open(MODEL_INFO_PATH)) if os.path.exists(MODEL_INFO_PATH) else {}
    all_info[model_key] = info
    json.dump(all_info, open(MODEL_INFO_PATH, "w"), indent=2)
    print(f"  [{model_key}] devices={devices} offloaded_layers={info['n_layers_offloaded']} "
          f"compute_dtype={info['quantization_config'].get('bnb_4bit_compute_dtype')}")
    assert info["n_layers_offloaded"] == 0, "Layers were offloaded to CPU/disk; energy would be inflated. Stop."
    return model, tokenizer


def format_chat_prompt(tokenizer, user_content):
    try:
        return tokenizer.apply_chat_template([{"role": "user", "content": user_content}],
                                             tokenize=False, add_generation_prompt=True)
    except Exception:
        return user_content


def generate_text(model, tokenizer, prompt, max_new_tokens):
    prompt = format_chat_prompt(tokenizer, prompt)
    inputs = tokenizer(prompt, return_tensors="pt", add_special_tokens=False).to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False,
                             temperature=None, top_p=None, pad_token_id=tokenizer.eos_token_id)
    new_ids = out[0][inputs["input_ids"].shape[1]:]
    text = tokenizer.decode(new_ids, skip_special_tokens=True).strip()
    generate_text.last_n_input = int(inputs["input_ids"].shape[1])
    return text, int(new_ids.shape[0])

## 8. Task data

In [ ]:
from datasets import load_dataset

def _extract_gsm8k_answer(answer_field):
    m = re.search(r"####\s*([\-0-9,\.]+)", answer_field)
    return m.group(1).replace(",", "").strip() if m else None

_DATA_CACHE = {}
def load_task_data(task_name, n_samples=N_SAMPLES_PER_TASK, seed=SEED):
    if task_name in _DATA_CACHE: return _DATA_CACHE[task_name]
    if task_name == "summarization":
        ds = load_dataset("abisee/cnn_dailymail", "3.0.0", split="test", cache_dir=DRIVE_CACHE_DIR)
        ds = ds.shuffle(seed=seed).select(range(n_samples))
        data = [{"input": ex["article"], "reference": ex["highlights"]} for ex in ds]
    elif task_name == "classification":
        ds = load_dataset("fancyzhx/ag_news", split="test", cache_dir=DRIVE_CACHE_DIR)
        ds = ds.shuffle(seed=seed).select(range(n_samples))
        names = ds.features["label"].names
        data = [{"input": ex["text"], "reference": names[ex["label"]]} for ex in ds]
    elif task_name == "math":
        ds = load_dataset("openai/gsm8k", "main", split="test", cache_dir=DRIVE_CACHE_DIR)
        ds = ds.shuffle(seed=seed).select(range(n_samples))
        data = [{"input": ex["question"], "reference": _extract_gsm8k_answer(ex["answer"])} for ex in ds]
    else:
        raise ValueError(task_name)
    _DATA_CACHE[task_name] = data
    return data

## 9. Prompts (English) and self-confidence scoring (fixed parser)

In [ ]:
def build_base_prompt(task_name, model_input):
    if task_name == "summarization":
        return ("Summarize the following news article in 2-3 sentences. "
                "Write the summary in English.\n\n" + model_input[:3000])
    elif task_name == "classification":
        return ("Classify the following news text into exactly ONE of these four categories: "
                "World, Sports, Business, Sci/Tech.\n\n"
                "Respond with ONLY the category name (World, Sports, Business or Sci/Tech). "
                "Do not add any explanation or other words.\n\n"
                f"Text: {model_input[:2000]}\n\nCategory:")
    elif task_name == "math":
        return ("Solve the following math problem step by step. "
                "On the last line, write your final answer in the format 'Answer: <number>'.\n\n"
                f"Problem: {model_input}")
    raise ValueError(task_name)


def build_retry_prompt(base_prompt, previous_output, confidence):
    return (f"{base_prompt}\n\n"
            f"Your previous answer was: {previous_output}\n"
            f"That answer received a low confidence score ({confidence}/10). "
            "Produce a more accurate and clearer answer:")


SCORE_RE = re.compile(r"\b(10|[1-9])\b")

def parse_confidence(raw):
    m = SCORE_RE.search(raw)
    if m: return int(m.group(1)), False
    return 5, True          # default on failure, flagged


def self_confidence_score(model, tokenizer, task_name, model_input, model_output):
    judge_prompt = (
        f"Below is an output produced for a {task_name} task.\n\n"
        f"INPUT: {model_input[:1500]}\n\n"
        f"OUTPUT: {model_output}\n\n"
        "Rate the correctness and quality of this output on a scale from 1 (very poor) to 10 (perfect). "
        "Reply with ONLY a single number. Do not write anything else.")
    raw, _ = generate_text(model, tokenizer, judge_prompt, JUDGE_MAX_NEW_TOKENS)
    score, failed = parse_confidence(raw)
    return max(1, min(score, 10)), raw, failed

## 10. Agentic loop

In [ ]:
def agentic_loop(model, tokenizer, task_name, sample, threshold):
    base_prompt = build_base_prompt(task_name, sample["input"])
    prompt = base_prompt
    max_toks = TASK_MAX_TOKENS[task_name]
    conf_hist, outputs, judge_raw, ntoks, failed_flags = [], [], [], [], []
    for turn in range(MAX_EXTRA_TURNS + 1):
        output, n_new = generate_text(model, tokenizer, prompt, max_toks)
        conf, raw, failed = self_confidence_score(model, tokenizer, task_name, sample["input"], output)
        outputs.append(output); ntoks.append(n_new); conf_hist.append(conf)
        judge_raw.append(raw); failed_flags.append(failed)
        if conf >= threshold or turn == MAX_EXTRA_TURNS:
            break
        prompt = build_retry_prompt(base_prompt, output, conf)
    return {"final_output": outputs[-1], "n_turns": len(outputs), "confidence_history": conf_hist,
            "per_turn_outputs": outputs, "judge_raw_history": judge_raw,
            "n_new_tokens_history": ntoks, "truncated_any": any(n >= max_toks for n in ntoks),
            "parse_failed_any": any(failed_flags), "n_parse_failed": sum(failed_flags)}

## 11. Energy measurement

In [ ]:
def run_with_energy_tracking(model, tokenizer, task_name, sample, threshold, run_id):
    tracker = EmissionsTracker(project_name=run_id, output_dir=RESULTS_DIR, log_level="error",
                               save_to_file=False, measure_power_secs=0.1)
    tracker.start(); t0 = time.time()
    res = agentic_loop(model, tokenizer, task_name, sample, threshold)
    elapsed = time.time() - t0
    emissions = tracker.stop()
    def _kwh(attr):
        v = getattr(tracker, attr, None)
        return getattr(v, "kWh", None) if v is not None else None
    res.update({"elapsed_sec": elapsed, "emissions_kg_co2eq": emissions,
                "energy_kwh": _kwh("_total_energy"), "gpu_energy_kwh": _kwh("_total_gpu_energy"),
                "cpu_energy_kwh": _kwh("_total_cpu_energy"), "ram_energy_kwh": _kwh("_total_ram_energy")})
    return res

## 12. Scoring

In [ ]:
rouge = evaluate.load("rouge")

def score_task(task_name, prediction, reference):
    if task_name == "summarization":
        return {"rougeL": rouge.compute(predictions=[prediction], references=[reference])["rougeL"]}
    if task_name == "classification":
        return {"correct": int(reference.strip().lower() in prediction.strip().lower())}
    if task_name == "math":
        nums = re.findall(r"[\-]?\d[\d,]*\.?\d*", prediction)
        if not nums or reference is None: return {"correct": 0}
        try:
            return {"correct": int(abs(float(nums[-1].replace(",", "")) - float(reference)) < 1e-4)}
        except ValueError:
            return {"correct": 0}
    raise ValueError(task_name)

## 13. Component-wise measured loop

In [ ]:
def measured_call(fn, *args):
    """Run fn(*args) inside its own CodeCarbon interval; return (result, energy dict)."""
    tr = EmissionsTracker(project_name="comp", output_dir=RESULTS_DIR, log_level="error",
                          save_to_file=False, measure_power_secs=0.1)
    tr.start(); t0 = time.time()
    res = fn(*args)
    el = time.time() - t0
    tr.stop()
    def _k(a):
        v = getattr(tr, a, None); return getattr(v, "kWh", None) if v is not None else None
    return res, {"energy_kwh": _k("_total_energy"), "gpu_energy_kwh": _k("_total_gpu_energy"),
                 "cpu_energy_kwh": _k("_total_cpu_energy"), "ram_energy_kwh": _k("_total_ram_energy"),
                 "elapsed_sec": el}


def component_loop(model, tokenizer, task_name, sample, threshold):
    """Same control flow as the v4 agentic loop, but every call is measured separately."""
    base_prompt = build_base_prompt(task_name, sample["input"])
    prompt = base_prompt
    max_toks = TASK_MAX_TOKENS[task_name]
    rows, outputs, conf_hist = [], [], []
    for turn in range(MAX_EXTRA_TURNS + 1):
        (output, n_new), e = measured_call(generate_text, model, tokenizer, prompt, max_toks)
        rows.append({"component": f"gen{turn+1}", "in_tokens": generate_text.last_n_input, "out_tokens": n_new,
                     "truncated": int(n_new >= max_toks), **e})
        (conf, raw, failed), e = measured_call(self_confidence_score, model, tokenizer, task_name, sample["input"], output)
        rows.append({"component": f"judge{turn+1}", "in_tokens": generate_text.last_n_input, "out_tokens": None,
                     "truncated": 0, "confidence": conf, "judge_raw": raw, "parse_failed": int(failed), **e})
        outputs.append(output); conf_hist.append(conf)
        if conf >= threshold or turn == MAX_EXTRA_TURNS:
            break
        prompt = build_retry_prompt(base_prompt, output, conf)
    return rows, outputs, conf_hist


def _load_done():
    if os.path.exists(COMP_CSV):
        df = pd.read_csv(COMP_CSV)
        return df.to_dict("records"), set(zip(df["model"], df["task"], df["sample_idx"], df["repeat"]))
    return [], set()


def run_components(model_keys=list(MODEL_REGISTRY.keys()), tasks=ACTIVE_TASKS, tau=7):
    rows, done = _load_done()
    for model_key in model_keys:
        todo = [(task, i, r) for task in tasks for r in range(REPEATS) for i in range(N_SAMPLES_PER_TASK)
                if (model_key, task, i, r) not in done]
        if not todo:
            print(f"[{model_key}] complete, skipping."); continue
        print(f"\n=== Loading {model_key} ===")
        model, tokenizer = load_model(model_key)
        generate_text(model, tokenizer, build_base_prompt("classification", "Warm-up text."), 5)  # untracked warm-up
        for task in tasks:
            data = load_task_data(task)
            for r in range(REPEATS):
                remaining = [i for i in range(N_SAMPLES_PER_TASK) if (model_key, task, i, r) not in done]
                if not remaining:
                    continue
                print(f"  task={task} repeat={r}: {len(remaining)} samples")
                for count, idx in enumerate(remaining, 1):
                    sample = data[idx]
                    comp_rows, outputs, conf_hist = component_loop(model, tokenizer, task, sample, tau)
                    metrics = score_task(task, outputs[-1], sample["reference"])
                    first_metrics = score_task(task, outputs[0], sample["reference"])
                    for cr in comp_rows:
                        rows.append({"model": model_key, "model_category": MODEL_REGISTRY[model_key]["category"],
                                     "task": task, "threshold": tau, "sample_idx": idx, "repeat": r,
                                     "n_turns": len(outputs), "confidence_history": json.dumps(conf_hist),
                                     "final_metric": metrics.get("correct", metrics.get("rougeL")),
                                     "first_turn_metric": first_metrics.get("correct", first_metrics.get("rougeL")),
                                     "timestamp": datetime.now(timezone.utc).isoformat(), **cr})
                    done.add((model_key, task, idx, r))
                    if count % FLUSH_EVERY == 0 or count == len(remaining):
                        pd.DataFrame(rows).to_csv(COMP_CSV, index=False)
                        print(f"     [{count}/{len(remaining)}] saved")
        del model, tokenizer
        gc.collect(); torch.cuda.empty_cache(); torch.cuda.synchronize()
        if MODEL_REGISTRY[model_key]["category"] == "large":
            cache_path = os.path.join(os.environ["HF_HOME"], "hub", "models--" + MODEL_REGISTRY[model_key]["hf_id"].replace("/", "--"))
            if os.path.exists(cache_path):
                shutil.rmtree(cache_path, ignore_errors=True)
    print("\nComponent run complete:", COMP_CSV)

run_components()


## 14. Quick check

In [ ]:
df = pd.read_csv(COMP_CSV)
print(len(df), "rows;", df.groupby("model").sample_idx.nunique().to_dict())
e = df.groupby(["task", "model_category", "component"])["energy_kwh"].mean().unstack("component")
print("\nMean energy per call (kWh):\n", e.round(7))
g1 = df[df.component == "gen1"].groupby(["task", "model"])["energy_kwh"].mean()
j1 = df[df.component == "judge1"].groupby(["task", "model"])["energy_kwh"].mean()
print("\nJudge share of first-turn energy:\n", (j1 / (g1 + j1)).unstack("task").round(3))
cv = (df[df.component == "gen1"].groupby(["task", "model", "sample_idx"])["energy_kwh"]
      .agg(lambda x: x.std() / x.mean()).groupby(["task", "model"]).median())
print("\nMedian CV of gen1 energy across repeats:\n", cv.unstack("task").round(3))
